# Can a Databricks-written table be good enough without V-Order?

Direct Lake models over **the same TPC-DS rows**, differing only in who wrote the parquet
and how:

| arm | writer | V-Order | layout |
|---|---|---|---|
| `tpcds_sf{sf}_cluster` | **Databricks**, parquet-mr, Photon off | **no** | the SAME config as `default` plus `CLUSTER BY` on the date key, declared on an empty table and then appended, so the rows are PLACED by the key on write. This is how an ordering reaches the files on this write path: a `df.orderBy()` does not, because Optimized Writes plans a repartition above it and `EliminateSorts` deletes the sort |
| `tpcds_sf{sf}_default` | **Databricks**, parquet-mr, Photon off | **no** | **configuration only**: plain `saveAsTable` -- optimize-write bin 4096 MiB, ~6M-row row groups (one per file), dictionary kept, ZSTD, no ordering, no `OPTIMIZE` |
| `tpcds_sf{sf}_partition` | **Databricks**, parquet-mr, Photon off | **no** | the paper's Fabric arm reproduced minus V-Order: `partitionBy` the date key and nothing else. One file per date, ~143k-row row groups, both recipe row caps inert far above it. `partition` vs `vorder` is V-Order ALONE at that geometry |
| `tpcds_sf{sf}_vorder` | **Fabric Spark** | **yes** | the paper's arm, four things at once: V-Order, ZSTD, optimize write, **partitioned by the date key** and **Z-ordered on the address key**. The partition lands ONE FILE PER PARTITION -- 1,823 files of ~143k rows at SF100, so ~1,800 Direct Lake segments |
| `tpcds_sf{sf}_vonly` | **Fabric Spark** | **yes** | V-Order and nothing else: **no partition, no sort, no `OPTIMIZE`**. Its geometry lands on `default`'s (measured 5.8M rows/file against 5.6M, 45 files against 46), so `vonly` vs `default` is V-Order encoding ALONE, with everything else held equal |
| `tpcds_sf{sf}_duckdb` | **delta_rs** (delta-rs, via duckrun) | **no** | the third writer, reading the `default` rows back and rewriting them under duckrun's fixed 0.4.68 profile: a 4M-row row-group CEILING inside a 256 MB file roll (so every file ends on a truncated group), a 32 MB dictionary page limit, **SNAPPY**. `SORTED BY AUTO` -- duckrun profiles the data and picks the ORDER BY itself, to minimise modelled in-memory bytes rather than to prune. The key it chose is in the build notebook's output |

V-Order cannot be produced outside Fabric. The question is whether the row-group geometry, the
dictionary and the ordering get close enough that it does not matter.

The `duck*` arms are SNAPPY where every other arm is ZSTD -- duckrun's notebook API cannot set the
codec -- so their **bytes** are not comparable to any other arm's. Their **encodings** are, because
dictionary coverage is classified from the parquet footer rather than from size.

The pair that settles it is **`default` vs `vonly`**: same rows, same geometry, no ordering and no
partition on either side, so the only thing left between them is the writer's encoding. The second
chart below draws exactly those two. `vorder` answers a different question -- what the paper
measured -- and `cluster` answers a third, whether an ordering that reaches the files pays.

The protocol is the **paper's own**: each arm's model is created once, measured by **three
back-to-back load tests**, then deleted. `run_index` 1 is the first touch of a model created
minutes earlier -- it pays the transcode inside the suite, exactly as their Run 1 does; 2 and 3 are
the same suite again on the same model.

So there are two comparisons here, and they answer different questions:

* **steady state (runs 2-3)** -- what the parquet layout costs a resident model. This is the arm
  vs arm number.
* **run 1 -> 2 -> 3** -- whether an arm warms up at all. That curve is the paper's own headline:
  at SF100/20 users its V-Order arm went 99.5 / 5.0 / 6.2 s while its mirrored arm went
  114.8 / 142.7 / 166.9 s and never warmed. A fixed mirroring tax cannot produce those two shapes;
  layout can.

The suite number quoted throughout is theirs: the **sum of the per-query medians over the 15
visual queries**, medians taken per query first (a pooled median would weight a query by how many
readers finished it).

One asymmetry to keep in mind on run 1: the mirrored arm's files live in the Databricks metastore's
storage (North Europe) while this capacity is in West Central US; the V-Order arm reads OneLake in
region. That costs the mirrored arm a transatlantic round trip on every first read, and nothing at
steady state, where no storage is on the path.

In [ ]:
!pip install -q duckrun --upgrade
notebookutils.session.restartPython()

In [ ]:
import duckdb
import notebookutils

lh_name = "tpcds_bench"          # holds perfresults3
results_table = "perfresults3"   # the THREE-RUN protocol's table. `perfresults` is the old
                                 # probe + one-warm-pass protocol: different definition of a first
                                 # pass, never to be unioned with this one. Point this at a smoke
                                 # table to read a rehearsal run instead.
MIRROR_ITEM = "01b539f3-4a9d-45ef-b1ef-0ba59552eb21"   # mirrored Azure Databricks catalog
VORDER_LH = "tpcds_vorder"       # the Fabric Spark V-Order copy
# Scale factor is a PARAMETER, not a constant: nothing here is regenerated to move between
# scale factors. Override `sf` in the parameters cell (or from a pipeline) and the schemas,
# the model names and the stats targets all follow.
sf = 100
VORDER_SCHEMA = f"tpcds_sf{sf}"
N_QUERIES = 24                   # all-24-or-out; a partial suite is not a run
RUNS = 3                         # load tests per model lifetime -- the paper's Run column
SLICER = (3, 7, 8, 11, 15, 16, 17, 18, 21)   # the 9 slicer queries, reported apart

ws_id = notebookutils.runtime.context["currentWorkspaceId"]
lh_id = notebookutils.lakehouse.get(lh_name)["id"]
# GUIDs, never friendly names: this tenant has OneLake friendly-name support disabled, so
# `<ws>/tpcds_vorder.Lakehouse` is refused outright while `<ws-guid>/<item-guid>` works everywhere.
vorder_id = notebookutils.lakehouse.get(VORDER_LH)["id"]
delta_path = (f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}/"
              f"Tables/dbo/{results_table}")
print(delta_path)

In [ ]:
# Every virtual user appends its own small file, so 20 users x 3 runs x N arms leaves the table a
# heap of tiny files. Compacting is housekeeping, not analysis -- it changes no number below.
#
# The vacuum keeps a WEEK. It used to keep nothing (retention_hours=0,
# enforce_retention_duration=False) and that destroyed the table on 2026-09-08: vacuum deletes every
# parquet under the folder that the CURRENT snapshot does not reference, which includes a file a
# concurrent `write_deltalake(mode="append")` has already uploaded but not yet committed. The
# appender then commits an `add` for a file that is gone, and from that moment every full read of
# perfresults3 -- including the `count_rows()` guard that every run notebook opens with -- fails
# with a 404 on the missing blob. That is what the 7-day default and `enforce_retention_duration`
# exist to prevent, so both are back on. Old files now linger for a week instead of being reclaimed
# at once; the table is a few hundred KB of results, so that costs nothing.
#
# If the table is ALREADY broken, `DeltaTable(delta_path).repair(dry_run=False)` drops the dangling
# references (FSCK) and the rows in the deleted file are lost -- the all-24-or-out filter below
# discards the affected run anyway.
from deltalake import DeltaTable
_dt = DeltaTable(delta_path)
_dt.optimize.compact()
_dt.vacuum(retention_hours=168, dry_run=False)
print("compacted")

In [ ]:
# The comparable result set. Two filters, both applied to EVERY run including run 1:
#   all-24-or-out  a run is one (loadtest_id, thread_id, iteration) and counts only if all 24
#                  queries SUCCEEDED -- a failed query still writes its row, so counting rows would
#                  let a run that errored on three queries pass as complete.
#   short rung     a rung that asked for 20 users and got 8 measured 8-way concurrency.
import pyarrow as pa
from deltalake import DeltaTable

perf = pa.table(DeltaTable(delta_path).to_pyarrow_table())
duckdb.register("perf", perf)

# Arm from the last '_' token of the model name -- exact equality, no pattern matching.
# Every one of these names a LAYOUT. Which engine wrote the files is not a variable in this
# experiment -- VertiPaq reads parquet from any producer -- and labelling the arms "dbx" vs "Fabric"
# made the charts read as a producer comparison, which is the exact misreading the whole thing
# exists to prevent. V-Order stays in the names because it IS a layout property (an encoding plus an
# in-row-group sort), not an engine.
# A REMOVED arm's rows are filtered out below rather than relabelled: nothing is ever deleted from
# perfresults3, so without the filter those rows reach the charts through ELSE as a model name
# pretending to be an arm. Removed from the repo on
# 2026-09-08. Its rows stay in perfresults3 -- nothing is deleted from the results table -- and are
# excluded here so they cannot be read as a fifth layout.
ARM_DEFAULT = "Databricks: the recipe (6M rows per file)"
# The same unordered write at 8M. `default` is 6M with one row group per file and this is 8M with
# one per file, so segment SIZE is the only thing between them -- the test of whether size is a
# lever without an ordering. Prediction on the record, and MEASURED: within noise of `default`.
ARM_DEFAULTF8 = "Databricks: 8M rows per file"
# WITHDRAWN 2026-09-09, and it is the arm that earned its own withdrawal. It held 6M row groups TWO
# to a 12M-row file, which only `parquet.block.row.count.limit` can express, and it measured
# identical to one group per file. So the key bought nothing -- and it needs parquet-java 1.16,
# where every older runtime accepts it and ignores it. The key left the recipe and this arm left
# with it. Its rows stay in perfresults3 and its chunks in the footer export, filtered by name.
ARM_DEFAULT2RG = "Databricks: 6M groups, 2 per file (withdrawn)"
# The clustered arm again, in SNAPPY and with the 128 MB byte target the first clustered build
# predates -- the two things the delta_rs arm had and ours did not. Its pair is `cluster`, and
# the question is COLD: delta_rs transcodes in 17.7 s where our clustered arm takes 36.9.
ARM_CLUSTERSN = "Databricks: clustered by date, snappy"
ARM_CLUSTER, ARM_VORDER = "Databricks: clustered by date", "Fabric: partition per date + Z-order + V-Order"
ARM_VONLY = "Fabric: V-Order"
ARM_PARTITION = "Databricks: partition per date"
# The delta_rs arms name the WRITER first, unlike every label above. On the other arms the writer is
# a constant within its cloud and the layout is the variable; here the layout is deliberately held
# at what the Databricks arms already ran and the WRITER is what changed, so a label that said only
# "4M groups" would hide the one thing the arm exists to vary.
ARM_DUCKDB = "delta_rs: auto sort key"
ARM_DUCKSORT = "delta_rs: sorted per date"

# A removed arm leaves rows in `perfresults3` AND chunks in the `layout_stats` footer export --
# nothing is ever deleted from either -- so it has to be filtered by NAME in BOTH places, and this
# is the one list. Drop an arm from the repo without adding it here and it comes back through the
# ELSE above as a model name pretending to be an arm, or as a timing-less row in the layout table.
# Never re-use a token that has been here.
WITHDRAWN_ARMS = ("sort", "nosort4m", "default2rg")
_WITHDRAWN = ", ".join(f"'{a}'" for a in WITHDRAWN_ARMS)
duckdb.sql(f"""
    CREATE OR REPLACE TABLE scanned AS
    SELECT *,
           CASE str_split(model, '_')[-1]
                WHEN 'default' THEN '{ARM_DEFAULT}'
                WHEN 'defaultf8' THEN '{ARM_DEFAULTF8}'
                WHEN 'default2rg' THEN '{ARM_DEFAULT2RG}'
                WHEN 'cluster' THEN '{ARM_CLUSTER}'
                WHEN 'clustersn' THEN '{ARM_CLUSTERSN}'
                WHEN 'vorder' THEN '{ARM_VORDER}'
                WHEN 'vonly' THEN '{ARM_VONLY}'
                WHEN 'partition' THEN '{ARM_PARTITION}'
                WHEN 'duckdb' THEN '{ARM_DUCKDB}'
                WHEN 'ducksort' THEN '{ARM_DUCKSORT}'
                ELSE model END AS arm,
           CAST(regexp_extract(model, 'sf([0-9]+)', 1) AS INTEGER) AS sf
    FROM perf
    WHERE str_split(model, '_')[-1] NOT IN ({_WITHDRAWN})   -- removed arms; rows stay in the table
""")
duckdb.sql(f"""
    CREATE OR REPLACE TABLE runs AS
    WITH complete AS (
        SELECT loadtest_id, thread_id, iteration FROM scanned
        GROUP BY 1,2,3
        HAVING COUNT(DISTINCT CASE WHEN error IS NULL THEN query_number END) = {N_QUERIES}
    ), full_rung AS (
        SELECT s.loadtest_id FROM scanned s JOIN complete USING (loadtest_id, thread_id, iteration)
        GROUP BY s.loadtest_id
        HAVING COUNT(DISTINCT s.thread_id) = MAX(s.concurrent_threads)
    )
    SELECT s.* FROM scanned s
    JOIN complete USING (loadtest_id, thread_id, iteration)
    JOIN full_rung USING (loadtest_id)
    WHERE s.error IS NULL
""")
kept = duckdb.sql("SELECT count(*) FROM runs").fetchone()[0]
total = duckdb.sql("SELECT count(*) FROM scanned").fetchone()[0]
print(f"{kept:,} of {total:,} rows are comparable")

In [ ]:
# THE table. One row per (sf, arm, run) -- filter it however you like.
#
# TWO-STEP MEDIAN, as the paper's analysis does it: median across readers WITHIN a query first,
# then combine queries. `suite_s` is the SUM of those per-query medians over the 15 visual queries,
# so it is the query time ONE reader spends making a full pass -- not an average, and not the load
# test's wall clock (readers run concurrently and there is `delay_sec` think time between queries).
# It is the paper's own metric: its 99.5 / 5.0 / 6.2 (V-Order) and 114.8 / 142.7 / 166.9 (mirrored)
# are exactly this number.
#
# Read ACROSS the runs: run 1 is the first touch of a model created minutes earlier and pays the
# transcode inside the suite; runs 2-3 are the same suite on the same resident model.
duckdb.sql("""
    CREATE OR REPLACE TABLE tops AS
    SELECT sf, max(concurrent_threads) AS top FROM runs GROUP BY 1
""")
SFS = [r[0] for r in duckdb.sql("SELECT sf FROM tops ORDER BY sf").fetchall()]
RUN_INDEXES = [r[0] for r in duckdb.sql("SELECT DISTINCT run_index FROM runs ORDER BY 1").fetchall()]

# Step one of the two-step, and the base the chart draws from.
duckdb.sql("""
    CREATE OR REPLACE TABLE per_q AS
    SELECT r.sf, r.arm, r.run_index, r.query_number,
           any_value(r.visual_name) AS visual_name,
           median(r.duration) AS d,
           count(*) AS execs
    FROM runs r JOIN tops t ON r.sf = t.sf AND r.concurrent_threads = t.top
    GROUP BY ALL
""")

# `summary` is THE table, and the chart below plots straight out of it -- one number cannot then
# disagree with the other.
#
# suite_s is the two-step per-query statistic (the paper's metric). p50 / p95 / max are NOT: they
# come straight off the RAW executions -- every reader's every visual query in the run, one pool of
# 20 x 15 durations -- so the three of them describe one distribution: typical, tail, worst.
#
# They used to be computed over the 15 per-query medians instead (`median(d)`, `median(d95)`), and
# `p95_ms` in particular was then a median-of-p95s, nobody's percentile: the V-Order cold run read
# 1,300 ms while 5% of its executions were over 24,100 ms and its worst was 26,518 ms.
#
# load_tests / models come from a SEPARATE aggregate, joined at group level. Joining row-level
# `runs` into the per-query query fanned every row out by its 480 executions and multiplied
# suite_s by 480 (it read 33,047 where the answer was 68.8).
#
# load_tests > 1 means the arm was benchmarked more than once and this row POOLS those sessions.
# Fine when they agree; check RES_PERQUERY before trusting a row where they might not.
duckdb.sql(f"""
    CREATE OR REPLACE TABLE summary AS
    WITH q AS (
        SELECT sf, arm, run_index,
               round(sum(d), 1) AS suite_s,
               count(*) AS queries
        FROM per_q WHERE query_number NOT IN {SLICER}
        GROUP BY ALL
    ), e AS (
        SELECT r.sf, r.arm, r.run_index, max(r.concurrent_threads) AS users,
               round(1000 * quantile_cont(r.duration, 0.5)) AS p50_ms,
               round(1000 * quantile_cont(r.duration, 0.95)) AS p95_ms,
               -- The single worst execution anywhere in the run: one query, one reader, the whole
               -- transcode landing on it.
               round(1000 * max(r.duration)) AS max_ms,
               count(*) AS execs,
               count(DISTINCT r.loadtest_id) AS load_tests,
               count(DISTINCT r.model_id) AS models
        FROM runs r JOIN tops t ON r.sf = t.sf AND r.concurrent_threads = t.top
        WHERE r.query_number NOT IN {SLICER}
        GROUP BY ALL
    )
    SELECT q.sf, q.arm, e.users, q.run_index, q.suite_s, e.p50_ms, e.p95_ms, e.max_ms,
           q.queries, e.execs, e.load_tests, e.models
    FROM q JOIN e USING (sf, arm, run_index)
    ORDER BY sf, arm, run_index
""")
display(duckdb.sql("SELECT * FROM summary").df())

In [ ]:
# TWO CHARTS PER SCALE FACTOR, both plotting `suite_s` straight out of `summary` -- the same column
# the table shows, so a number on a picture is always a number in that table.
#
#   1. this cell: every arm measured -- the whole warming curve
#   2. next cell: `default` against `vonly`, the pair that isolates V-Order
#
# LOG y, and it has to be: run 1 carries the whole transcode and is 10-30x runs 2-3, so on a linear
# axis every warm point collapses onto the baseline. The point labels carry the real numbers, so
# nothing depends on reading a gridline.
import matplotlib.pyplot as plt
import numpy as np

d = duckdb.sql("SELECT sf, arm, run_index, suite_s FROM summary").df()

ARMS = [ARM_DEFAULT, ARM_DEFAULTF8,
        ARM_CLUSTER, ARM_CLUSTERSN, ARM_PARTITION, ARM_VORDER, ARM_VONLY,
        ARM_DUCKDB, ARM_DUCKSORT]
# The legend names the LAYOUT, not the engine -- that is what these charts are about. Naming them
# "dbx" vs "Fabric" would read as a producer comparison, which is the misreading the whole
# experiment exists to prevent.
# Row-group size leads every label where the recipe SETS it, because that is the variable: one
# parquet row group is one VertiPaq segment, so "6M groups" and "~143k groups" are ~46 segments per
# fact against ~1,800.
#
# The cluster arm's label deliberately carries NO row-group size. It used to say "6M-row groups,
# clustered by date" and that was a claim nobody had checked; measured at SF100 the clustered write
# produced 2.05M-row groups on store_sales and 1.11M on catalog_sales. The row caps never fire on a
# clustered write -- the clustering exchange picks the file, and therefore the group -- so any
# number in this label would be a number the recipe did not choose and does not hold across tables
# or scale factors. See LEARNING.md, *The clustered SF100 arm, measured both ways*.
SHORT = {ARM_DEFAULT: "Databricks: the recipe -- 6M rows per file, 1 row group",
         ARM_DEFAULTF8: "Databricks: 8M rows per file, 1 row group",
         ARM_DEFAULT2RG: "Databricks: 6M x2 per file (withdrawn)",
         ARM_CLUSTER: "Databricks: clustered by date (write sizes the groups)",
         ARM_CLUSTERSN: "Databricks: clustered by date, 128 MB files, snappy",
         ARM_VONLY: "Fabric: V-Order",
         ARM_PARTITION: "Databricks: partition per date",
         ARM_VORDER: "Fabric: partition per date + Z-order + V-Order",
         ARM_DUCKDB: "delta_rs: auto sort key",
         ARM_DUCKSORT: "delta_rs: sorted per date"}
# The two V-Order pairs share a hue so the comparison is visible before the legend is read:
# orange/purple is the recipe's geometry without/with V-Order, teal/red is partition-per-date.
# The three delta_rs arms share a blue, for the same reason: on those the WRITER is the variable,
# so the family has to read as one before anyone gets to the legend. The four unordered Databricks
# unordered Databricks arms share an orange: one write, two geometries.
COLOR = {ARM_DEFAULT: "#f58518", ARM_DEFAULTF8: "#c2571a", ARM_DEFAULT2RG: "#ffb266",
         ARM_CLUSTER: "#54a24b", ARM_CLUSTERSN: "#2f6b28",
         ARM_VONLY: "#b279a2", ARM_PARTITION: "#72b7b2", ARM_VORDER: "#e45756",
         ARM_DUCKDB: "#4c78a8", ARM_DUCKSORT: "#7fa8c9"}


def suite_for(sf_, arm, run_):
    hit = d[(d.sf == sf_) & (d.arm == arm) & (d.run_index == run_)]
    return float(hit.suite_s.iloc[0]) if len(hit) else np.nan


def draw(sf_, arms, title):
    users = duckdb.sql(f"SELECT top FROM tops WHERE sf = {sf_}").fetchone()[0]
    vals = {(a, r): suite_for(sf_, a, r) for a in arms for r in RUN_INDEXES}
    # Only the arms actually measured get a line. A flat line through absent runs would claim a
    # measurement that does not exist.
    shown = [a for a in arms if any(not np.isnan(vals[(a, r)]) for r in RUN_INDEXES)]
    if not shown:
        print(f"SF{sf_}: nothing to chart for {title}")
        return
    present = [v for v in vals.values() if not np.isnan(v)]
    lo, hi = min(present), max(present)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_yscale("log")
    ax.set_ylim(lo / 2.2, hi * 4)      # room for the point labels
    for arm in shown:
        xs = [i for i, r in enumerate(RUN_INDEXES) if not np.isnan(vals[(arm, r)])]
        ys = [vals[(arm, RUN_INDEXES[i])] for i in xs]
        ax.plot(xs, ys, marker="o", markersize=8, linewidth=2.5, color=COLOR[arm],
                label=SHORT[arm])
        for x, y in zip(xs, ys):
            ax.annotate(f"{y:,.1f}s", (x, y), textcoords="offset points", xytext=(0, 9),
                        ha="center", fontsize=9, color=COLOR[arm], fontweight="bold")

    ax.set_xticks(range(len(RUN_INDEXES)))
    ax.set_xticklabels([f"run {r}" for r in RUN_INDEXES], fontsize=12)
    ax.set_xlim(-0.35, len(RUN_INDEXES) - 0.65)
    ax.set_ylabel("suite_s -- one reader's full pass, 15 visual queries, log scale")
    ax.set_title(f"SF{sf_}: {title} -- three load tests over ONE model, {users} readers")
    ax.set_xlabel("run 1 = first touch of a model created minutes earlier  |  "
                  "runs 2-3 = the same suite again, same model")
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3, which="both")
    plt.tight_layout()
    plt.show()


# `sf_`, not `sf`: `sf` is this notebook's parameter and a loop variable would overwrite it.
for sf_ in SFS:
    draw(sf_, ARMS, "every arm")

In [ ]:
# BEST AGAINST BEST, and nothing else on the axes.
#
# `clustered by date` is the best layout Databricks can reach from CONFIGURATION plus one CLUSTER BY
# -- no V-Order, because Databricks has none. `partition per date + Z-order + V-Order` is the best
# Fabric produces, and it is the paper's own arm. That pair is the question this whole repo exists
# to answer: can a Databricks team, writing Delta with a pasted config, land a table that Direct Lake
# reads as well as one Fabric wrote for itself?
#
# `delta_rs sorted per date` is the third writer's best: delta-rs ordering the facts on the SAME
# date key the clustered arm uses. Three engines, each at the best layout it can reach.
#
# READ IT AS THREE LAYOUTS, NOT THREE PRODUCTS. Each line is the best LAYOUT one writer reached, and
# the chart below it is what stops this from being a league table: `vonly` is Fabric's own writer,
# V-Order and all, with NO layout work, and it is the WORST arm here -- ~11.7 s in steady state
# against these three at ~2.3-2.5. So no writer is good on its own and none is disqualified by its
# encoding; what separates the lines is the layout each was asked for.
#
# `default` and `partition` stay off: they are mechanism controls that isolate one variable each, and
# they are drawn below so neither chart is read as the other.
for sf_ in SFS:
    draw(sf_, [ARM_CLUSTER, ARM_VORDER, ARM_DUCKSORT],
         "best per writer: Databricks, Fabric, delta-rs")
# The control, on its own axes so it cannot be mistaken for a fourth contender.
for sf_ in SFS:
    draw(sf_, [ARM_VONLY, ARM_VORDER], "V-Order with no layout work vs V-Order with it")

In [ ]:
# The mechanism pairs. Each holds everything constant but ONE thing, which is what makes them
# readable at all -- and which is why they are not on the best-against-best chart above.
for sf_ in SFS:
    # Same rows, same writer settings in every respect that shapes the files -- no ordering, no
    # partition, no sort, no OPTIMIZE on either side -- and, measured, the same geometry: 45 files
    # of 5.8M rows against 46 of 5.6M. So the ONLY thing left between these two lines is whether the
    # parquet was written with V-Order.
    #
    # `vorder` is deliberately absent HERE: it partitions the facts by the date key, landing 1,823
    # files of ~143k rows, ~1,800 segments against these two arms' ~45 -- 40x. On this chart it would
    # answer a question about partitioning while looking like an answer about V-Order.
    draw(sf_, [ARM_DEFAULT, ARM_VONLY], "V-Order or not, everything else equal")
    # The same question at the other geometry, and THE decisive pair: partitioned by the date key
    # partitioned by the date key on BOTH sides, so the file count and the elimination are the
    # same. TWO things differ, not one: V-Order, and the row order INSIDE each file -- Fabric's
    # OPTIMIZE ZORDER interleaves (measured 11 runs per distinct address key), the Databricks arm
    # does not order at all (1.02, i.e. what the source already was). An attempt to match that with
    # `sortWithinPartitions` never reached the parquet and was removed on 2026-09-08; see
    # LEARNING.md, *The within-partition sort never reached the files*. So this pair BOUNDS the
    # V-Order effect, it does not isolate it.
    draw(sf_, [ARM_PARTITION, ARM_VORDER], "partition per date on both: V-Order and in-file order differ")
    # And the geometry question with the encoding held out of it entirely.
    draw(sf_, [ARM_DEFAULT, ARM_PARTITION], "6M-row groups vs partition per date, neither V-Ordered")
    # SEGMENT SIZE WITH NOTHING TO ELIMINATE. Same writer, same config but one number, no ordering on
    # either side, one row group per file on both: 6M against 8M. Every arm in this project with
    # small groups and no useful ordering is slow (the withdrawn 4M arm was no better than this 6M,
    # duckdb at 2.4M is 14.2 s, vonly at 1.5-2.9M is 11.7 s) while every arm with small groups AND
    # an ordering is fast (cluster 2.0M, ducksort 2.1-2.8M, both ~2.5 s). That says the ordering is
    # the lever and the size is not. This pair tests the other direction: if size were the lever,
    # going BIGGER should hurt. If the two lines sit together, it is not.
    # The withdrawn `default2rg` was the third line here: 6M groups TWO per 12M-row file, which
    # isolated groups-per-file at a fixed segment. It tied, the row-group key left the recipe with
    # it, and it is filtered out by name -- so this pair is now the whole geometry question, and
    # `default` is what every other arm should be read against.
    draw(sf_, [ARM_DEFAULT, ARM_DEFAULTF8],
         "unordered segment size, one row group per file: 6M against 8M")
    # WHO PICKS THE SORT KEY. Both arms are ordered and neither is V-Ordered; what differs is who
    # chose the key. `cluster` sorts on the date surrogate because 23 of the 24 captured queries
    # filter on it -- a key chosen for PRUNING. `duckdb` sorts on whatever duckrun's recommender
    # picked, which optimises modelled in-memory BYTES and is free to take up to 4 columns. Since
    # only the FIRST key eliminates row groups, the interesting outcome is a split: `duckdb` smaller
    # and cheaper cold, `cluster` faster on the date-filtered queries.
    #
    # NOT a single-variable pair, and it must not be read as one -- the writers, the row-group sizes
    # and the compression all differ too (delta_rs writes SNAPPY where every other arm is ZSTD, so
    # BYTES between these lines are not comparable). Read it with the key that `build_duckdb`
    # printed and with the layout table below, never on its own.
    draw(sf_, [ARM_CLUSTER, ARM_DUCKDB], "a key chosen for pruning vs a key duckrun picked")
    # THE ORDERING PAIR, and the cleanest one the delta_rs family gives: `ducksort` orders on the
    # SAME date key `cluster` clusters on, so the key is held and the WRITER is what changes -- a
    # Databricks clustering exchange against a delta-rs sort. Still not single-variable (row-group
    # size and SNAPPY vs ZSTD differ), but the ordering is finally the same on both sides, which
    # `duckdb` could never give: AUTO picked a key chosen to shrink memory, not to prune.
    draw(sf_, [ARM_CLUSTER, ARM_DUCKSORT], "the same date key, two writers")
    # WHAT delta_rs HAD THAT WE DID NOT. `ducksort` is the best COLD arm in the project -- 17.7 s
    # against the clustered arm's 36.9 -- and the two differences that could account for it are the
    # CODEC (snappy decompresses far faster than zstd) and the DICTIONARY, which our clustered
    # catalog_sales lost 19.6 % of its bytes to at 1.11M rows per group. `clustersn` is the same
    # CLUSTER BY with both addressed: snappy, and `delta.targetFileSize` = 128 MB to lift the groups
    # into the 2-5M band where the dictionary holds. Both are TABLE PROPERTIES, so the write is
    # still `df.write.saveAsTable()` and the recipe is still the config plus one CLUSTER BY.
    draw(sf_, [ARM_CLUSTER, ARM_CLUSTERSN, ARM_DUCKSORT], "same date key: zstd, snappy, delta_rs")
    # Partition per date, two writers: parquet-mr holds ~50% of fact bytes as dictionary at that
    # ~143k-row geometry and V-Order holds 100%. That gap is the dictionary bracket, and it is the
    # one thing V-Order was measured to be the only answer to.
    draw(sf_, [ARM_PARTITION, ARM_VORDER],
         "partition per date, two writers: parquet-mr, V-Order")

In [ ]:
# WHAT THE ARMS ARE SITTING ON -- one table, every scale factor, next to the charts it explains.
#
# Read straight off the RAW footer export. `layout_stats` writes `chunks_<arm>.parquet` per scale
# factor -- one row per column chunk per row group, exactly what `parquet_metadata` returned -- and
# this cell recomputes geometry, dictionary and ordering from it. Its summary CSVs are deliberately
# not read: they are one arm-list and one sf per run, already aggregated, and a second copy of a
# number is a second thing that can be stale.
#
# EVERY COLUMN, for a reader who has seen none of this before. An `arm` is one way of writing the
# SAME TPC-DS data: same rows, same schema, same queries -- only the parquet layout differs. The
# charts compare how fast Power BI reads each one. This table is what they are sitting on.
#
#   WHAT IT IS
#   sf              TPC-DS scale factor. 100 is roughly 100 GB of source data, 1000 ten times that.
#   arm             the layout, named exactly as the chart legends name it -- and prefixed with the
#                   engine that wrote it, which is why there is no separate writer column. The
#                   engine is NOT the variable under test: VertiPaq reads parquet from any producer,
#                   what the charts measure is LAYOUT, and every writer here appears at both the
#                   fast and the slow end. Read a row by its layout columns, not by its prefix.
#   table           the two fact tables. Everything else is a small dimension and is left out.
#
#   HOW BIG
#   rows            rows in the table. Identical across arms at one sf -- same data, laid out
#                   differently -- so a difference here means an arm was built wrong.
#   table_gb        compressed column bytes on disk. Not the same as memory: VertiPaq re-encodes.
#   files           parquet files. More, smaller files is not automatically worse; see row_groups.
#   file_mb         average file. The recipe aims at 128 MB on a clustered write.
#
#   HOW IT IS SHAPED  -- this is the part that moves the charts
#   row_groups      row groups across the whole table. ONE PARQUET ROW GROUP BECOMES ONE VERTIPAQ
#                   SEGMENT, so this is the segment count Direct Lake ends up with.
#   rows_per_group  rows in the average one. Direct Lake wants 1M..16M: below it there are too many
#                   segments to manage, above it a segment is too coarse to skip. When a row lands
#                   outside that window the `note` column says so.
#
#   HOW IT IS ENCODED
#   dict_pct        % of bytes the parquet writer kept dictionary-encoded. Whatever is NOT
#                   dictionary gets its dictionary REBUILT by VertiPaq the first time the column is
#                   read -- time and memory on every cold read, which is what run 1 pays for.
#
#   HOW IT IS ORDERED
#   key             the column the arm was ordered or partitioned on, if any.
#   overlaps        neighbouring row groups whose key ranges intersect. Where they overlap, a filter
#                   cannot rule either one out, so the engine reads both.
#   sorting         the verdict in words. `eliminable` = disjoint ranges, whole row groups can be
#                   skipped. `partitioned on the key` is STRONGER still: one key value per file, so
#                   nothing can overlap at all, and it outranks any measured overlap count.
#
#   WHAT IS SURPRISING
#   note            ONLY what the other columns cannot tell you, or where an arm did not do what its
#                   name says. Not a description of the arm. An ordinary row has an EMPTY note, so a
#                   note means look here. Three things can appear: a CLUSTER BY that did not sort
#                   the rows at this scale, a Z-order that cannot help elimination because the
#                   partition already put one date in each file, and a SORTED BY AUTO that ordered
#                   the table on a key no query filters on. Anything a reader can get by reading
#                   another column stays OUT -- `rows_per_group` against the window above, `files`
#                   against `row_groups`, and
#                   `dict_pct`, are theirs to read.
#
#   WHAT IT COST  -- THREE load tests are run back to back over ONE model, then it is deleted. Run 1
#                    is the first touch of a model created minutes earlier, so it pays the whole
#                    Delta-to-memory transcode inside the run. Runs 2 and 3 are the same suite again
#                    on the same resident model. Reading ACROSS the runs is the point: whether an arm
#                    warms up at all is the paper's own headline.
#   full_runs       how many COMPLETE load tests feed this row, over all its runs: a load test counts
#                   only if every reader finished all 24 queries and the rung got the users it asked
#                   for. 3 is one model lifetime; 9 is three lifetimes pooled into the same medians.
#   run<N>_total_s  TOTAL SECONDS for the whole 15-query suite, for ONE reader, on that run. Built
#                   per query as the median across the 20 readers, then summed over the 15 queries.
#                   So it is a total, not a per-query average, and not the load test's wall clock,
#                   which is shorter because the readers run concurrently with think time between
#                   queries. These are exactly the points plotted on the charts above. Compare arms
#                   on the LAST run; that is steady state.
#   cold_*/warm_*   the same two ends, for the per-EXECUTION statistics below: cold is run 1, warm is
#                   the last run. These are MILLISECONDS and describe ONE query, not the suite.
#   *_p50_ms        the typical single execution, over every reader's every query in that run.
#   *_p95_ms        95th percentile of a SINGLE query execution, over the same pool. The tail a user
#                   actually waits on when the model is busy.
#   *_max_ms        the single worst execution anywhere in the run -- one query, one reader, the
#                   whole transcode landing on it. Cold max is usually the ugliest number here.
#                   All of these come from the SAME `summary` table the charts plot, so a number here
#                   can never disagree with one up there. Blank means the arm was not benchmarked.
#
# Dimensions are excluded: every one is a single file and a single row group at every arm, so they
# say nothing about the charts and would triple the row count.
import duckrun

RG_LO, RG_HI = 1_000_000, 16_000_000            # Direct Lake's usable row-group window
# The first ordering key of each fact -- the column whose row-group ranges say whether the ordering
# reached the files. Same map as layout_stats.
KEY = {"store_sales": "ss_sold_date_sk", "catalog_sales": "cs_sold_date_sk"}
_KV = ", ".join(f"('{t}', '{c}')" for t, c in KEY.items())
_KCOLS = ", ".join(f"'{c}'" for c in KEY.values())

# Same lakehouse layout_stats writes to -- lh_name is already tpcds_bench, so ws_id and lh_id from
# the bootstrap cell are the address and no GUID lookup is needed. `bench.con` is duckrun's own
# DuckDB connection and carries the OneLake credential, so the footers are read in place: nothing
# is downloaded and no local temp folder is left behind.
bench = duckrun.connect(f"{ws_id}/{lh_id}", name="bench")
CHUNKS = (f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}/"
          "Files/layout_stats/sf*/chunks_*.parquet")

# The scale factor is the FOLDER, which is what makes one glob cover every sf at once: layout_stats
# runs one sf at a time and each run drops its own sf<n>/ directory. Projected down here, on the
# remote connection, so what crosses into this notebook is the columns below and nothing else --
# min/max are cast and kept for the KEY columns only, which is the whole of what they are read for.
# `.arrow()` hands back a RecordBatchReader on some DuckDB builds and a Table on others, and a
# reader is consumed by the first query that touches it. to_arrow_table / fetch_arrow_table is a
# materialised Table on every build.
#
# WITHDRAWN_ARMS is applied HERE as well as on the results table. A removed arm keeps its chunks in
# the export just as it keeps its rows in perfresults3, and this cell does not join to the timings
# -- it LEFT JOINs them -- so without the filter a withdrawn arm reaches the layout table anyway,
# as a row with real geometry and empty timings. Filtering at the read rather than in the CTE also
# keeps the chunk count printed below honest.
try:
    _rel = bench.con.sql(f"""
        SELECT CAST(regexp_extract(filename, 'sf([0-9]+)', 1) AS INTEGER) AS sf,
               arm, "table" AS tbl, file_name, row_group_id,
               row_group_num_rows AS rg_rows, path_in_schema AS col,
               total_compressed_size AS bytes, encodings,
               CASE WHEN path_in_schema IN ({_KCOLS})
                    THEN TRY_CAST(stats_min_value AS BIGINT) END AS lo,
               CASE WHEN path_in_schema IN ({_KCOLS})
                    THEN TRY_CAST(stats_max_value AS BIGINT) END AS hi
        FROM read_parquet('{CHUNKS}', filename = true, union_by_name = true)
        WHERE arm NOT IN ({_WITHDRAWN})
    """)
    _raw = (_rel.to_arrow_table() if hasattr(_rel, "to_arrow_table") else _rel.fetch_arrow_table())
except Exception as e:                                                      # noqa: BLE001
    _raw = None
    print(f"no footer export under {lh_name}: Files/layout_stats ({str(e)[:140]}).")
    print("Run the `layout_stats` notebook once per scale factor, then re-run this cell.")

if _raw is not None:
    duckdb.register("lay_chunks", _raw)
    print(f"{_raw.num_rows:,} column chunks over "
          f"{duckdb.sql('SELECT count(DISTINCT (sf, arm)) FROM lay_chunks').fetchone()[0]} "
          "scale-factor x arm combinations")

    # Cold and warm off `summary` -- the SAME suite_s the charts plot, so a number here can never
    # disagree with a number up there. min_by/max_by over run_index rather than a hardcoded 1 and 3:
    # a session that ran two runs, or four, still reads correctly.
    # One suite_s column PER RUN, so a row carries the arm's whole warming curve and the columns line
    # up one-for-one with the points on its chart line. Built from RUN_INDEXES, which the headline
    # cell discovered from the data -- a session that ran two runs, or four, gets two or four columns
    # rather than three that are half empty.
    # `run<N>_total_s`, spelled out: it is a TOTAL in SECONDS -- the whole 15-query suite for ONE
    # reader, per-query medians across the readers then summed. Not a per-query average, and not the
    # load test's wall clock, which is shorter because readers run concurrently with think time.
    _RUNS = ", ".join(f"max(suite_s) FILTER (WHERE run_index = {r}) AS run{r}_total_s"
                      for r in RUN_INDEXES) or "NULL::DOUBLE AS run_total_s"
    _RUNCOLS = ", ".join(f"t.run{r}_total_s" for r in RUN_INDEXES) or "t.run_total_s"
    _LAST = f"t.run{RUN_INDEXES[-1]}_total_s" if RUN_INDEXES else "t.run_total_s"
    duckdb.sql(f"""
        CREATE OR REPLACE TABLE lay_time AS
        SELECT sf, arm,
               -- Complete load tests pooled into this row, over every run index. `load_tests` in
               -- `summary` is per run, so the sum is the count of distinct surviving loadtest_ids.
               CAST(sum(load_tests) AS INTEGER) AS full_runs,
               {_RUNS},
               -- p95 and max are per-EXECUTION, off the raw pool of every reader's every query, so
               -- they only make sense at the two ends: the run that pays the transcode and the run
               -- that does not. min_by/max_by over run_index rather than a hardcoded 1 and 3.
               min_by(p50_ms, run_index) AS cold_p50_ms,
               min_by(p95_ms, run_index) AS cold_p95_ms,
               min_by(max_ms, run_index) AS cold_max_ms,
               max_by(p50_ms, run_index) AS warm_p50_ms,
               max_by(p95_ms, run_index) AS warm_p95_ms,
               max_by(max_ms, run_index) AS warm_max_ms
        FROM summary GROUP BY 1, 2
    """)

    # The chunks carry the RAW arm names; the charts carry the layout labels. Relabel or a row
    # cannot be matched to the line it explains. ELSE arm, so a name this build does not know
    # (a withdrawn arm, or the pre-rename `layout`) still shows rather than becoming NULL.
    LAY_ARM = f"""CASE arm WHEN 'default' THEN '{ARM_DEFAULT}'
                           WHEN 'defaultf8' THEN '{ARM_DEFAULTF8}'
                           WHEN 'default2rg' THEN '{ARM_DEFAULT2RG}'
                           WHEN 'cluster' THEN '{ARM_CLUSTER}'
                           WHEN 'clustersn' THEN '{ARM_CLUSTERSN}'
                           WHEN 'partition' THEN '{ARM_PARTITION}'
                           WHEN 'vorder' THEN '{ARM_VORDER}'
                           WHEN 'vonly' THEN '{ARM_VONLY}' WHEN 'duckdb' THEN '{ARM_DUCKDB}'
                           WHEN 'ducksort' THEN '{ARM_DUCKSORT}'
                           ELSE arm END"""
    # `note` carries ONLY what is surprising -- what the row's own columns cannot tell you, or where
    # the arm did not do what its name says. It is NOT a description of the arm: `dict_pct`,
    # `overlaps` and `sorting` are right there and a reader can read them. An unremarkable row gets
    # an EMPTY note, and that is the point: a note means look here.
    #
    # Everything below is derived from this row, so nothing can go stale except the one design fact
    # -- that an ordering inside a one-file-per-date partition cannot help elimination, since the
    # partition already gives it -- and even that only fires on a row measured to be partitioned.
    _INPART_SORT = "'%Z-order%'"

    duckdb.sql(f"""
        CREATE OR REPLACE TABLE layout AS
        WITH c AS (SELECT * REPLACE ({LAY_ARM} AS arm) FROM lay_chunks),
             k(tbl, key) AS (VALUES {_KV}),
             -- A row group contributes one row per COLUMN, so the distinct (file, group) pairs come
             -- first: summing rg_rows over the raw chunks multiplies every count by the column count.
             rg AS (SELECT DISTINCT sf, arm, tbl, file_name, row_group_id, rg_rows FROM c),
             -- PARTITIONED ON THE KEY is the strongest elimination there is, and it is read off
             -- the PATH, not guessed: Hive writes the value into the directory name
             -- (`.../ss_sold_date_sk=2451742/part-0000...`), so one partition holds exactly one key
             -- value and no two files can overlap. Whether the column is ALSO inside the parquet is
             -- a writer detail and nothing else -- Fabric drops it, Databricks keeps it -- so the
             -- two partitioned arms used to get different verdicts for the same layout.
             part AS (
                 SELECT r.sf, r.arm, r.tbl,
                        max(CASE WHEN r.file_name LIKE '%/' || k.key || '=%' THEN 1 ELSE 0 END) = 1
                            AS partitioned
                 FROM rg r JOIN k ON r.tbl = k.tbl GROUP BY 1, 2, 3
             ),
             geom AS (
                 SELECT sf, arm, tbl, CAST(sum(rg_rows) AS BIGINT) AS "rows",
                        count(DISTINCT file_name) AS files,
                        count(*) AS row_groups,
                        CAST(round(avg(rg_rows)) AS BIGINT) AS rows_per_group
                 FROM rg GROUP BY 1, 2, 3
             ),
             -- On-disk size, from the column chunks rather than the row groups: bytes are a
             -- per-column-chunk figure, so they must be summed before the distinct-row-group step
             -- that `rg` does. This is compressed COLUMN data -- footers and page headers are not
             -- in it -- so it reads a little under what the Files listing shows.
             size AS (
                 SELECT sf, arm, tbl, sum(bytes) AS total_bytes,
                        sum(bytes) / count(DISTINCT file_name) AS bytes_per_file
                 FROM c GROUP BY 1, 2, 3
             ),
             -- Dictionary BY BYTES. `encodings` is a comma-separated list, so it is split and
             -- trimmed rather than matched with LIKE: a bare LIKE '%PLAIN%' also matches
             -- PLAIN_DICTIONARY and would report every dictionary chunk as a fallback.
             e AS (SELECT sf, arm, tbl, bytes,
                          list_transform(str_split(coalesce(encodings, ''), ','), x -> trim(x)) AS encs
                   FROM c),
             dict AS (
                 -- The same rule layout_stats uses: a dictionary encoding in the list, full stop.
                 -- Not "and no bare PLAIN" -- delta_rs lists PLAIN on every dictionary chunk, since
                 -- the dictionary page is PLAIN-encoded, and excluding those scored that writer at
                 -- zero. No parquet-mr chunk in this project has ever carried both.
                 SELECT sf, arm, tbl,
                        round(100.0 * sum(bytes) FILTER (
                            WHERE list_contains(encs, 'RLE_DICTIONARY')
                               OR list_contains(encs, 'PLAIN_DICTIONARY')) / sum(bytes), 1) AS dict_pct
                 FROM e GROUP BY 1, 2, 3
             ),
             -- One [lo, hi] per row group on the first key, for the arms that are NOT partitioned on
             -- it. Fabric's V-Order arm has no chunks here at all -- it drops the partition column
             -- from the file -- and does not need them: `part` above already settled it.
             kr AS (
                 SELECT c.sf, c.arm, c.tbl, any_value(c.lo) AS lo, any_value(c.hi) AS hi
                 FROM c JOIN k ON c.tbl = k.tbl AND c.col = k.key
                 WHERE c.lo IS NOT NULL
                 GROUP BY c.sf, c.arm, c.tbl, c.file_name, c.row_group_id
             ),
             -- Sorted by low bound: if a range starts before its predecessor ends, that pair overlaps
             -- and a filter landing in the overlap eliminates neither group.
             ov AS (
                 SELECT sf, arm, tbl, count(*) AS key_groups,
                        count(*) FILTER (WHERE prev_hi IS NOT NULL AND prev_hi > lo) AS overlaps
                 FROM (SELECT *, lag(hi) OVER (PARTITION BY sf, arm, tbl ORDER BY lo, hi) AS prev_hi
                       FROM kr)
                 GROUP BY 1, 2, 3
             )
        SELECT g.sf, g.arm, g.tbl AS "table",
               g."rows", round(z.total_bytes / 1073741824.0, 2) AS table_gb,
               g.files, round(z.bytes_per_file / 1048576.0, 1) AS file_mb,
               g.row_groups, g.rows_per_group,
               d.dict_pct, k.key,
               -- Zero by construction when the key is the partition column, not by measurement:
               -- every file in a partition carries the one value the directory names.
               CASE WHEN p.partitioned THEN 0 ELSE o.overlaps END AS overlaps,
               CASE WHEN p.partitioned      THEN 'partitioned on the key (one value per file)'
                    WHEN o.key_groups IS NULL THEN 'no min/max stats on the key'
                    WHEN o.overlaps = 0       THEN 'eliminable'
                    ELSE CAST(round(100.0 * o.overlaps / o.key_groups) AS INTEGER)
                         || '% of neighbours overlap' END AS sorting,
               t.full_runs, {_RUNCOLS},
               t.cold_p50_ms, t.cold_p95_ms, t.cold_max_ms,
               t.warm_p50_ms, t.warm_p95_ms, t.warm_max_ms,
               concat_ws('; ',
                   -- The arm is named for an ordering it did not deliver. At SF100 the clustered
                   -- write placed the rows and this is silent; at SF1000 it did not.
                   CASE WHEN g.arm = '{ARM_CLUSTER}' AND o.overlaps > 0
                        THEN 'CLUSTER BY did not sort the rows at this scale -- '
                             || CAST(round(100.0 * o.overlaps / o.key_groups) AS INTEGER)
                             || '% of neighbouring row groups still overlap' END,
                   -- The arm DID sort -- AUTO just sorted on a key nothing filters on, so the
                   -- overlap count beside it reads like a failed sort and is not one. Unconditional
                   -- for the arm: it is what `SORTED BY AUTO` is, not something that went wrong.
                   CASE WHEN g.arm = '{ARM_DUCKDB}'
                        THEN 'SORTED BY AUTO picked its own key -- it minimises modelled memory, '
                             || 'not pruning -- so the date key the queries filter on is unordered' END,
                   -- The Fabric arm's OPTIMIZE ZORDER did rewrite every file (measured: rows
                   -- interleaved on the address key), but it cannot buy elimination -- one date per
                   -- file already gives that. The Databricks partition arm has no in-file ordering
                   -- at all, so this note is the Fabric one's alone.
                   CASE WHEN p.partitioned AND g.arm LIKE {_INPART_SORT}
                        THEN 'the Z-order reorders rows inside each file; it cannot help '
                             || 'elimination -- one date per file already does' END
               ) AS note
        FROM geom g
        JOIN k ON g.tbl = k.tbl                       -- facts only, and it supplies the key
        LEFT JOIN dict d ON d.sf = g.sf AND d.arm = g.arm AND d.tbl = g.tbl
        LEFT JOIN ov   o ON o.sf = g.sf AND o.arm = g.arm AND o.tbl = g.tbl
        LEFT JOIN part p ON p.sf = g.sf AND p.arm = g.arm AND p.tbl = g.tbl
        LEFT JOIN size z ON z.sf = g.sf AND z.arm = g.arm AND z.tbl = g.tbl
        LEFT JOIN lay_time t ON t.sf = g.sf AND t.arm = g.arm
        ORDER BY g.sf, coalesce({_LAST}, 1e9), g.tbl
    """)

    # Coverage, said out loud. layout_stats takes an `arms` list and is run per sf, so PARTIAL is the
    # normal state -- and a join that quietly drops an arm would let an absent measurement read as an
    # absent difference.
    _gap = duckdb.sql("""
        SELECT sf, arm, 'charted, no footer export' AS missing FROM lay_time
        WHERE (sf, arm) NOT IN (SELECT sf, arm FROM layout)
        UNION ALL
        SELECT DISTINCT sf, arm, 'footer export, not charted' FROM layout WHERE warm_max_ms IS NULL
        ORDER BY 1, 3, 2
    """).df()
    if len(_gap):
        print("--- gaps. Run `layout_stats` at that sf with the arm in its `arms` list to fill one in.")
        display(_gap)

    print("--- one row per scale factor x arm x fact, best steady state first. Row groups of")
    print(f"    {RG_LO:,}..{RG_HI:,} rows are the ones Direct Lake wants; `sorting` is whether the")
    print("    ordering reached the files, so row groups can be eliminated at all.")
    display(duckdb.sql("SELECT * FROM layout").df())